# 01 — Data Profiling: Titik Panas Kebakaran Indonesia (NASA FIRMS VIIRS S-NPP)

Notebook ini memprofilkan data mentah titik panas (hotspot) dari NASA FIRMS,
sensor VIIRS S-NPP resolusi 375 m, untuk wilayah Indonesia. Tujuannya adalah
memahami struktur, kelengkapan, rentang nilai, dan potensi masalah kualitas
data sebelum tahap cleaning.

Seluruh pembacaan data menggunakan **Polars lazy API** (`scan_csv`) sehingga
data tidak dimuat ke memori sampai `.collect()` dipanggil. Profiling
pembanding dilakukan dengan **DuckDB `SUMMARIZE`**.

In [1]:
from pathlib import Path
import polars as pl
import duckdb

ROOT = Path("/home/jovyan/work")
RAW_DIR = ROOT / "data/raw/firms"
DATA_PATH = str(RAW_DIR / "*.csv")

## 1. Inventaris File

Data terdiri dari file CSV tahunan. Kolom `source_file` ditambahkan
saat pembacaan untuk menghitung jumlah baris per file.

In [2]:
files = sorted(RAW_DIR.glob("*.csv"))
total_mb = sum(f.stat().st_size for f in files) / 1e6
print(f"{len(files)} file, total {total_mb:,.1f} MB")

lf = pl.scan_csv(DATA_PATH, include_file_paths="source_file")

(lf.group_by("source_file")
   .agg(pl.len().alias("n_rows"))
   .sort("source_file")
   .collect())

13 file, total 270.3 MB


source_file,n_rows
str,u32
"""/home/jovyan/work/data/raw/fir…",318613
"""/home/jovyan/work/data/raw/fir…",257222
"""/home/jovyan/work/data/raw/fir…",555597
"""/home/jovyan/work/data/raw/fir…",875824
"""/home/jovyan/work/data/raw/fir…",117250
…,…
"""/home/jovyan/work/data/raw/fir…",91536
"""/home/jovyan/work/data/raw/fir…",68325
"""/home/jovyan/work/data/raw/fir…",57220


## 2. Missing Values

In [3]:
lf.select(pl.all().null_count()).collect()

latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,type,source_file
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## 3. Statistik Deskriptif Kolom Numerik

In [4]:
lf.select(pl.col(pl.Float64, pl.Int64)).describe()

statistic,latitude,longitude,bright_ti4,scan,track,acq_time,version,bright_ti5,frp,type
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",3.418464e6,3.418464e6,3.418464e6,3.418464e6,3.418464e6,3.418464e6,3.418464e6,3.418464e6,3.418464e6,3.418464e6
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",-2.762341,113.118685,331.286925,0.455154,0.467999,974.102889,2.0,294.086584,8.258314,0.080421
"""std""",3.414581,10.271383,16.824722,0.082027,0.108462,579.450311,0.0,8.763409,15.488921,0.39608
"""min""",-10.97676,95.01992,208.0,0.32,0.36,308.0,2.0,150.0,0.0,0.0
"""25%""",-3.91456,104.58615,318.82,0.39,0.38,541.0,2.0,289.74,2.34,0.0
"""50%""",-2.52338,112.32043,334.1,0.43,0.43,630.0,2.0,294.15,4.61,0.0
"""75%""",-0.23649,117.85938,341.75,0.5,0.54,1738.0,2.0,299.79,8.51,0.0
"""max""",5.8782,140.9774,367.0,0.8,0.78,2005.0,2.0,380.61,1951.3,3.0


## 4. Distribusi Kolom Kategorikal

In [5]:
for col in ["confidence", "daynight", "type", "satellite", "version"]:
    print(lf.group_by(col).agg(pl.len()).sort("len", descending=True).collect(), "\n")

shape: (3, 2)
┌────────────┬─────────┐
│ confidence ┆ len     │
│ ---        ┆ ---     │
│ str        ┆ u32     │
╞════════════╪═════════╡
│ n          ┆ 3089019 │
│ l          ┆ 211559  │
│ h          ┆ 117886  │
└────────────┴─────────┘ 

shape: (2, 2)
┌──────────┬─────────┐
│ daynight ┆ len     │
│ ---      ┆ ---     │
│ str      ┆ u32     │
╞══════════╪═════════╡
│ D        ┆ 2276431 │
│ N        ┆ 1142033 │
└──────────┴─────────┘ 

shape: (4, 2)
┌──────┬─────────┐
│ type ┆ len     │
│ ---  ┆ ---     │
│ i64  ┆ u32     │
╞══════╪═════════╡
│ 0    ┆ 3267691 │
│ 2    ┆ 88949   │
│ 1    ┆ 44227   │
│ 3    ┆ 17597   │
└──────┴─────────┘ 

shape: (1, 2)
┌───────────┬─────────┐
│ satellite ┆ len     │
│ ---       ┆ ---     │
│ str       ┆ u32     │
╞═══════════╪═════════╡
│ N         ┆ 3418464 │
└───────────┴─────────┘ 

shape: (1, 2)
┌─────────┬─────────┐
│ version ┆ len     │
│ ---     ┆ ---     │
│ i64     ┆ u32     │
╞═════════╪═════════╡
│ 2       ┆ 3418464 │
└─────────┴─────────┘ 


## 5. Validasi Rentang Nilai

Koordinat dibandingkan dengan perkiraan batas wilayah Indonesia
(lintang ±−11 s.d. 6, bujur ±95 s.d. 141).

In [7]:
lf.select(
    pl.col("acq_date").min().alias("tgl_awal"),
    pl.col("acq_date").max().alias("tgl_akhir"),
    pl.col("latitude").min().alias("lat_min"),
    pl.col("latitude").max().alias("lat_max"),
    pl.col("longitude").min().alias("lon_min"),
    pl.col("longitude").max().alias("lon_max"),
    pl.col("acq_time").min().alias("time_min"),
    pl.col("acq_time").max().alias("time_max"),
).collect()

tgl_awal,tgl_akhir,lat_min,lat_max,lon_min,lon_max,time_min,time_max
str,str,f64,f64,f64,f64,i64,i64
"""2012-01-20""","""2024-12-31""",-10.97676,5.8782,95.01992,140.9774,308,2005


## 6. Pemeriksaan Duplikasi

In [8]:
n_total = lf.select(pl.len()).collect().item()
n_unique = lf.drop("source_file").unique().select(pl.len()).collect().item()
print(f"Duplikat persis: {n_total - n_unique:,}")

Duplikat persis: 0


## 7. Profiling Pembanding dengan DuckDB `SUMMARIZE`

In [9]:
duckdb.sql(f"SUMMARIZE SELECT * FROM read_csv_auto('{DATA_PATH}')").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,latitude,DOUBLE,-10.97676,5.8782,1067596,-2.7623407038160974,3.4145806053131755,-3.959646470636064,-2.5188865320524294,-0.23900228581525,3418464,0.0
1,longitude,DOUBLE,95.01992,140.9774,1678994,113.11868450043963,10.271383189974582,104.62011700568314,112.31585917811606,117.88627024114336,3418464,0.0
2,bright_ti4,DOUBLE,208.0,367.0,6877,331.2869247387143,16.824722309904544,318.8838486916238,334.0890130035129,341.75419031832286,3418464,0.0
3,scan,DOUBLE,0.32,0.8,47,0.4551544202310062,0.08202656959911608,0.3901460426957618,0.43227565957639735,0.4999975540432517,3418464,0.0
4,track,DOUBLE,0.36,0.78,43,0.4679985982006262,0.10846153047650435,0.3757417800119139,0.4314799231004883,0.540877457658311,3418464,0.0
5,acq_date,DATE,2012-01-20,2024-12-31,5049,2017-01-19 14:57:36.364964,NaN,2014-09-07,2015-10-08,2019-09-10,3418464,0.0
6,acq_time,VARCHAR,0308,2005,525,NaN,NaN,NaN,NaN,NaN,3418464,0.0
7,satellite,VARCHAR,N,N,1,NaN,NaN,NaN,NaN,NaN,3418464,0.0
8,instrument,VARCHAR,VIIRS,VIIRS,1,NaN,NaN,NaN,NaN,NaN,3418464,0.0
9,confidence,VARCHAR,h,n,3,NaN,NaN,NaN,NaN,NaN,3418464,0.0


## 8. Ringkasan Temuan Profiling

### Ukuran dan cakupan
- Dataset terdiri dari **13 file CSV** (2012–2024) dengan total
  **3.418.464 baris** dan ukuran ±258 MB. Syarat tugas
  (> 1 juta baris) terpenuhi.
- Rentang tanggal: **20 Januari 2012 – 31 Desember 2024**. Data 2012
  tidak mencakup satu tahun penuh karena sensor VIIRS S-NPP baru mulai
  beroperasi awal 2012.
- Jumlah titik per tahun sangat bervariasi. Tahun **2015** memiliki
  875.824 titik (±25,6% dari seluruh data), sekitar 7,5 kali lipat
  tahun 2016 (117.250). Tahun 2023 (253.388) juga meningkat tajam
  dibanding 2020–2022. Hipotesis awal: variasi ini berkaitan dengan
  tahun-tahun kering; akan diuji pada tahap EDA.

### Kelengkapan dan duplikasi
- **Tidak ada missing value** pada seluruh 15 kolom.
- **Tidak ada duplikat persis** antarbaris.

### Kolom konstan
- `satellite` (selalu `N`), `instrument` (selalu `VIIRS`), dan
  `version` (selalu `2`) hanya memiliki satu nilai unik sehingga
  tidak informatif dan akan dihapus saat cleaning.

### Validitas nilai
- Koordinat berada di rentang lintang −10,98 s.d. 5,88 dan bujur
  95,02 s.d. 140,98; seluruhnya berada di dalam wilayah Indonesia.
- `frp` (Fire Radiative Power) sangat menceng ke kanan: median 4,61 MW,
  rata-rata 8,26 MW, maksimum 1.951,3 MW. Nilai ekstrem kemungkinan
  merupakan kebakaran besar yang nyata, bukan kesalahan data, sehingga
  akan ditandai (flag), bukan dihapus.
- Nilai minimum `frp` adalah 0. Jumlah baris dengan `frp == 0`: 119.

### Kolom kategorikal
- `type`: 95,6% bertipe 0 (kebakaran vegetasi). Sisanya bertipe 1
  (gunung api aktif, 44.227), 2 (sumber panas statis lain, 88.949),
  dan 3 (lepas pantai, 17.597). Tipe 1–3 bukan kebakaran hutan/lahan
  dan perlu dipisahkan dari analisis utama.
- `confidence`: `n` (nominal) 90,4%, `l` (low) 6,2%, `h` (high) 3,4%.
- `daynight`: siang 66,6%, malam 33,4%.

### Karakteristik waktu akuisisi
- `acq_time` hanya berkisar 03:08–20:05 UTC. Hal ini karena S-NPP
  adalah satelit orbit polar yang melintasi Indonesia sekitar dua kali
  sehari (±13.30 dan ±01.30 waktu lokal). Konsekuensinya, data **tidak
  dapat** digunakan untuk menganalisis jam terjadinya kebakaran dalam
  sehari; analisis temporal difokuskan pada skala harian, bulanan,
  musiman, dan tahunan.
- Deteksi malam (±17:00–20:00 UTC) jatuh pada dini hari **tanggal
  berikutnya** dalam waktu lokal Indonesia, sehingga tanggal perlu
  dikonversi ke waktu lokal saat cleaning.

### Perbedaan inferensi tipe Polars vs DuckDB
- Polars membaca `acq_date` sebagai String dan `acq_time` sebagai
  Int64 (nol di depan hilang, misalnya `0308` → `308`).
- DuckDB membaca `acq_date` sebagai DATE dan `acq_time` sebagai
  VARCHAR (`'0308'`).
- Karena inferensi otomatis tidak konsisten antar-engine, tipe kolom
  akan ditetapkan secara eksplisit pada tahap cleaning.

### Rencana tindak lanjut (Milestone 2)
1. Tetapkan schema eksplisit dan buat kolom datetime UTC serta waktu lokal.
2. Filter `type == 0` untuk analisis kebakaran vegetasi.
3. Keluarkan deteksi `confidence == "l"` dari analisis utama.
4. Hapus kolom konstan.
5. Tandai outlier `frp` dengan metode IQR tanpa menghapusnya.
6. Tandai tahun 2012 sebagai tahun tidak lengkap.
7. Validasi hasil dan simpan dalam format Parquet.